In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Bengali Handwritten Character Recognition
#  Adapted from the Devanagari (DHCD) version.
#
#  Transfer rationale
#  ──────────────────
#  Bengali and Devanagari share the same Brahmic ancestor and both use the
#  shirorekha (top headline).  Characters hang from the headline rather than
#  sitting on a baseline, so the horizontal-stroke scaffold branch (1×5 conv)
#  is directly meaningful.  The only structural changes vs. the Devanagari
#  version are:
#    • num_classes  : 46 → 50   (Bengali: 11 vowels + 39 consonants)
#    • Dataset path : DHCD      → CMATERdb 3.1.2  (or BanglaLekha-Isolated,
#                                  or any folder-per-class layout)
#    • Dataset name : DHCD      → CMATER / BanglaLekha (comments updated)
#  Everything else – architecture, augmentation, schedule, callbacks –
#  is unchanged.
# ─────────────────────────────────────────────────────────────────────────────

# importing necessary dependencies
import os, time, random, json
import numpy as np
import urllib.request, zipfile

# Silence TF info / warning messages; keep only errors.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
#     Only the fields marked  ← CHANGED  differ from the Devanagari version.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ← CHANGED: 50 classes for Bengali
    #   Standard breakdown: 11 vowels (স্বরবর্ণ) + 39 consonants (ব্যঞ্জনবর্ণ) = 50
    #   Adjust if your dataset uses a different subset (e.g. CMATERdb 3.1.2
    #   has 50 classes; BanglaLekha-Isolated has 84 including compound chars).
    "num_classes":      50,
    "image_size":       32,      # resize every image to 32×32 px
    "batch_size":       64,
    "epochs":           50,
    "lr":               5e-4,    # peak learning rate for cosine schedule
    "weight_decay":     1e-4,    # AdamW L2 regularisation
    "label_smoothing":  0.05,    # prevents over-confident softmax outputs
    "val_split":        0.1,     # fraction of training data held for validation

    # ← CHANGED: point to your Bengali dataset root.
    #   The directory must follow the same folder-per-class layout as DHCD:
    #     data_dir/
    #       Train/
    #         class_01/  *.png
    #         class_02/  *.png
    #         …
    #       Test/
    #         class_01/  *.png
    #         …
    #
    #   Recommended datasets (folder layout compatible):
    #     • CMATERdb 3.1.2  – 50 classes, ~12 000 train / ~3 000 test
    #       https://code.google.com/archive/p/cmater-biomedical/
    #     • BanglaLekha-Isolated  – 84 classes (basic + compound glyphs)
    #       https://data.mendeley.com/datasets/hf6sf8zrkc/2
    #     • NumtaDB  – digits only (10 classes) — use a different num_classes
    #       https://www.kaggle.com/datasets/BengaliAI/numta
    #
    #   Kaggle path example (CMATERdb):
    #     "/kaggle/input/cmaterdb/CMATERdb3.1.2"
    "data_dir":    "/kaggle/input/datasets/nayefshuvo/cmaterdb3-1-2/BasicFinalDatabase",

    "results_dir": "./results",
}

os.makedirs(CFG["results_dir"], exist_ok=True)
NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
#     Identical to the Devanagari version – just points at the Bengali folder.
# ─────────────────────────────────────────────────────────────────────────────
if os.path.exists(CFG["data_dir"]):
    print("[INFO] Bengali dataset already present – skipping download.")
else:
    raise FileNotFoundError(
        f"[ERROR] Dataset not found at {CFG['data_dir']}.\n"
        "Please download CMATERdb 3.1.2 (or BanglaLekha-Isolated) and "
        "extract it so the path above resolves to a Train/ + Test/ tree."
    )

# Load as RGB (3-channel) so TF's BMP decoder doesn't choke on 1-channel BMPs.
# tf.image.decode_bmp only accepts 0, 3 or 4 channels; "grayscale" forces 1
# and crashes on any BMP in the dataset.  We convert to grayscale ourselves
# immediately after loading via rgb_to_grayscale().
train_full = keras.utils.image_dataset_from_directory(
    os.path.join(CFG["data_dir"], "Train"),
    image_size=(IMG, IMG), batch_size=None,
    color_mode="rgb", label_mode="int", seed=SEED,
)
test_ds_raw = keras.utils.image_dataset_from_directory(
    os.path.join(CFG["data_dir"], "Test"),
    image_size=(IMG, IMG), batch_size=None,
    color_mode="rgb", label_mode="int", seed=SEED,
)

def to_grayscale(img, lbl):
    """Convert RGB tensor (H, W, 3) → grayscale (H, W, 1) using ITU-R 601 weights."""
    img = tf.image.rgb_to_grayscale(img)   # (H, W, 1), still uint8/float
    return img, lbl

train_full  = train_full.map(to_grayscale,  num_parallel_calls=AUTOTUNE)
test_ds_raw = test_ds_raw.map(to_grayscale, num_parallel_calls=AUTOTUNE)

total   = tf.data.experimental.cardinality(train_full).numpy()
n_val   = max(1, int(total * CFG["val_split"]))
n_train = total - n_val
train_ds_raw = train_full.take(n_train)
val_ds_raw   = train_full.skip(n_train)

# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    """Scale pixels from [0, 255] → [-1, 1]."""
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Light stochastic augmentation applied only during training.
    Pad-then-crop gives a random translation effect.
    Bengali characters are very sensitive to vertical stroke alignment,
    so we keep augmentations mild (no rotation to preserve the headline).
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    return img, lbl

def to_onehot(img, lbl):
    """Convert integer class index → one-hot vector."""
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
train_ds = (
    train_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    val_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    test_ds_raw
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    test_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: (batched)")

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20

    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║", "bold", "white"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n", "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS  (unchanged from Devanagari version)
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    """GELU activation – smoother than ReLU, better gradients in deep nets."""
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    """Standard pre-activation residual block."""
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    """
    DenseNet-inspired residual block.
    Three sequential residual blocks with dense connections, then a
    1×1 bottleneck and stride-2 depthwise downsampling.
    """
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    """Squeeze-and-Excitation (SE) channel attention."""
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    """Lightweight capsule-like routing module. O(n) cost, no dynamic routing."""
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Bengali stroke topology encoder.
    Uses the same asymmetric kernels as the Devanagari version because
    Bengali likewise has prominent horizontal (শিরোরেখা / shirorekha) and
    vertical stroke components.  The 1×5 branch captures the headline;
    the 5×1 branch captures vertical strokes such as those in ট, ঠ, ড, ঢ.
    Dilated convolutions add semi-local context for compound vowel matras.
    """
    h  = layers.Conv2D(64, (1, 5), padding="same", use_bias=False, activation=gelu)(x)
    v  = layers.Conv2D(64, (5, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1 = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d2 = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    out = layers.Concatenate()([h, v, d1, d2])
    out = layers.BatchNormalization()(out)
    out = layers.GlobalAveragePooling2D()(out)
    out = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    """
    Cross-scale transformer bridge (CSTB).
    Pools three encoder-scale feature maps to tokens and applies multi-head
    self-attention so fine-grained and coarse features interact.
    """
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
#     Renamed "our_model-Net" → "WhatNet-Bengali" for clarity.
#     num_classes is now 50 (passed in from CFG).
#     All architectural decisions carry over unchanged.
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_bengali(num_classes: int = 50, image_size: int = 32) -> Model:
    """
    WhatNet-Bengali: WhatNet adapted for Bengali handwritten character recognition.

    Architecture overview (identical structure to the Devanagari version)
    ─────────────────────────────────────────────────────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path (texture)
      • Horizontal stroke scaffold (1×5 conv) – captures the Bengali শিরোরেখা
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (16×16 at 32-px input)
      enc2: 64→128   ( 8× 8)
      enc3: 128→256  ( 4× 4)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      • Cross-scale transformer bridge (CSTB)
      • Adaptive filter capsule (AFC)
      • Stroke topology module (STM) – horizontal + vertical + dilated branches
      • Gated fusion → final MLP + layer norm → logits (50 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t       = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t       = layers.BatchNormalization()(t)
    t       = layers.Activation(gelu)(t)

    # Bengali শিরোরেখা scaffold: horizontal asymmetric conv
    s       = layers.Conv2D(32, (1, 5), padding="same", use_bias=False)(inp)
    s       = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(8)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Decoder head ──────────────────────────────────────────────────────
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)

    afc_in  = layers.GlobalAveragePooling2D()(enc3)
    afc_in  = layers.Add()([afc_in, cstb_out])
    afc_out = adaptive_filter_capsule(afc_in, K)

    stgm_out = stroke_topology_module(enc3, 256)

    combined    = layers.Concatenate()([stgm_out, afc_out])
    gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    fused       = layers.Concatenate()([stgm_scaled, afc_out])

    x   = layers.Dense(512)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="WhatNet-Bengali")


MODELS_TF = {
    "WhatNet-Bengali": lambda: build_whatnet_bengali(NUM_CLASSES, IMG),
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = sum(1 for _ in train_ds)
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs  [Bengali]", "bold", "white"))
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }

print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "bengali_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results  → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "bengali_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Bengali benchmark complete.\n", "green", "bold"))

[INFO] Bengali dataset already present – skipping download.
Found 12000 files belonging to 50 classes.
Found 3000 files belonging to 50 classes.
[INFO] Train: 10,800 | Val: 1,200 | Test: (batched)


InvalidArgumentError: {{function_node __wrapped__IteratorGetNext_output_types_2_device_/job:localhost/replica:0/task:0/device:CPU:0}} Error in user-defined function passed to ParallelMapDatasetV2:30 transformation with iterator: Iterator::Root::Prefetch::BatchV2::Shuffle::ParallelMapV2::ParallelMapV2::ParallelMapV2::FiniteTake::ParallelMapV2::Prefetch::ParallelMapV2: Input size should match (header_size + row_size * abs_height) but they differ by 1
	 [[{{node decode_image/DecodeImage}}]] [Op:IteratorGetNext] name: 

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Bengali Handwritten Character Recognition
#  Adapted from the Devanagari (DHCD) version.
#
#  Transfer rationale
#  ──────────────────
#  Bengali and Devanagari share the same Brahmic ancestor and both use the
#  shirorekha (top headline).  Characters hang from the headline rather than
#  sitting on a baseline, so the horizontal-stroke scaffold branch (1×5 conv)
#  is directly meaningful.  The only structural changes vs. the Devanagari
#  version are:
#    • num_classes  : 46 → 50   (Bengali: 11 vowels + 39 consonants)
#    • Dataset path : DHCD      → CMATERdb 3.1.2  (or BanglaLekha-Isolated,
#                                  or any folder-per-class layout)
#    • Dataset name : DHCD      → CMATER / BanglaLekha (comments updated)
#  Everything else – architecture, augmentation, schedule, callbacks –
#  is unchanged.
# ─────────────────────────────────────────────────────────────────────────────

# importing necessary dependencies
import os, time, random, json
import numpy as np
import urllib.request, zipfile

# Silence TF info / warning messages; keep only errors.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
#     Only the fields marked  ← CHANGED  differ from the Devanagari version.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ← CHANGED: 50 classes for Bengali
    #   Standard breakdown: 11 vowels (স্বরবর্ণ) + 39 consonants (ব্যঞ্জনবর্ণ) = 50
    #   Adjust if your dataset uses a different subset (e.g. CMATERdb 3.1.2
    #   has 50 classes; BanglaLekha-Isolated has 84 including compound chars).
    "num_classes":      50,

    "image_size":       32,      # resize every image to 32×32 px
    "batch_size":       64,
    "epochs":           50,
    "lr":               5e-4,    # peak learning rate for cosine schedule
    "weight_decay":     1e-4,    # AdamW L2 regularisation
    "label_smoothing":  0.05,    # prevents over-confident softmax outputs
    "val_split":        0.1,     # fraction of training data held for validation

    # ← CHANGED: point to your Bengali dataset root.
    #   The directory must follow the same folder-per-class layout as DHCD:
    #     data_dir/
    #       Train/
    #         class_01/  *.png
    #         class_02/  *.png
    #         …
    #       Test/
    #         class_01/  *.png
    #         …
    #
    #   Recommended datasets (folder layout compatible):
    #     • CMATERdb 3.1.2  – 50 classes, ~12 000 train / ~3 000 test
    #       https://code.google.com/archive/p/cmater-biomedical/
    #     • BanglaLekha-Isolated  – 84 classes (basic + compound glyphs)
    #       https://data.mendeley.com/datasets/hf6sf8zrkc/2
    #     • NumtaDB  – digits only (10 classes) — use a different num_classes
    #       https://www.kaggle.com/datasets/BengaliAI/numta
    #
    #   Kaggle path example (CMATERdb):
    #     "/kaggle/input/cmaterdb/CMATERdb3.1.2"
    # "data_dir":    "/kaggle/input/cmaterdb/CMATERdb3.1.2",
    "data_dir":    "/kaggle/input/datasets/nayefshuvo/cmaterdb3-1-2/BasicFinalDatabase",

    "results_dir": "./results",
}

os.makedirs(CFG["results_dir"], exist_ok=True)
NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
#     Identical to the Devanagari version – just points at the Bengali folder.
# ─────────────────────────────────────────────────────────────────────────────
if os.path.exists(CFG["data_dir"]):
    print("[INFO] Bengali dataset already present – skipping download.")
else:
    raise FileNotFoundError(
        f"[ERROR] Dataset not found at {CFG['data_dir']}.\n"
        "Please download CMATERdb 3.1.2 (or BanglaLekha-Isolated) and "
        "extract it so the path above resolves to a Train/ + Test/ tree."
    )

train_full = keras.utils.image_dataset_from_directory(
    os.path.join(CFG["data_dir"], "Train"),
    image_size=(IMG, IMG), batch_size=None,
    color_mode="grayscale", label_mode="int", seed=SEED,
)
test_ds_raw = keras.utils.image_dataset_from_directory(
    os.path.join(CFG["data_dir"], "Test"),
    image_size=(IMG, IMG), batch_size=None,
    color_mode="grayscale", label_mode="int", seed=SEED,
)

total   = tf.data.experimental.cardinality(train_full).numpy()
n_val   = max(1, int(total * CFG["val_split"]))
n_train = total - n_val
train_ds_raw = train_full.take(n_train)
val_ds_raw   = train_full.skip(n_train)

# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    """Scale pixels from [0, 255] → [-1, 1]."""
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Light stochastic augmentation applied only during training.
    Pad-then-crop gives a random translation effect.
    Bengali characters are very sensitive to vertical stroke alignment,
    so we keep augmentations mild (no rotation to preserve the headline).
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    return img, lbl

def to_onehot(img, lbl):
    """Convert integer class index → one-hot vector."""
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
train_ds = (
    train_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    val_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    test_ds_raw
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    test_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: (batched)")

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20

    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║", "bold", "white"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n", "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS  (unchanged from Devanagari version)
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    """GELU activation – smoother than ReLU, better gradients in deep nets."""
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    """Standard pre-activation residual block."""
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    """
    DenseNet-inspired residual block.
    Three sequential residual blocks with dense connections, then a
    1×1 bottleneck and stride-2 depthwise downsampling.
    """
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    """Squeeze-and-Excitation (SE) channel attention."""
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    """Lightweight capsule-like routing module. O(n) cost, no dynamic routing."""
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Bengali stroke topology encoder.
    Uses the same asymmetric kernels as the Devanagari version because
    Bengali likewise has prominent horizontal (শিরোরেখা / shirorekha) and
    vertical stroke components.  The 1×5 branch captures the headline;
    the 5×1 branch captures vertical strokes such as those in ট, ঠ, ড, ঢ.
    Dilated convolutions add semi-local context for compound vowel matras.
    """
    h  = layers.Conv2D(64, (1, 5), padding="same", use_bias=False, activation=gelu)(x)
    v  = layers.Conv2D(64, (5, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1 = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d2 = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    out = layers.Concatenate()([h, v, d1, d2])
    out = layers.BatchNormalization()(out)
    out = layers.GlobalAveragePooling2D()(out)
    out = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    """
    Cross-scale transformer bridge (CSTB).
    Pools three encoder-scale feature maps to tokens and applies multi-head
    self-attention so fine-grained and coarse features interact.
    """
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
#     Renamed "our_model-Net" → "WhatNet-Bengali" for clarity.
#     num_classes is now 50 (passed in from CFG).
#     All architectural decisions carry over unchanged.
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_bengali(num_classes: int = 50, image_size: int = 32) -> Model:
    """
    WhatNet-Bengali: WhatNet adapted for Bengali handwritten character recognition.

    Architecture overview (identical structure to the Devanagari version)
    ─────────────────────────────────────────────────────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path (texture)
      • Horizontal stroke scaffold (1×5 conv) – captures the Bengali শিরোরেখা
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (16×16 at 32-px input)
      enc2: 64→128   ( 8× 8)
      enc3: 128→256  ( 4× 4)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      • Cross-scale transformer bridge (CSTB)
      • Adaptive filter capsule (AFC)
      • Stroke topology module (STM) – horizontal + vertical + dilated branches
      • Gated fusion → final MLP + layer norm → logits (50 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t       = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t       = layers.BatchNormalization()(t)
    t       = layers.Activation(gelu)(t)

    # Bengali শিরোরেখা scaffold: horizontal asymmetric conv
    s       = layers.Conv2D(32, (1, 5), padding="same", use_bias=False)(inp)
    s       = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(8)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Decoder head ──────────────────────────────────────────────────────
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)

    afc_in  = layers.GlobalAveragePooling2D()(enc3)
    afc_in  = layers.Add()([afc_in, cstb_out])
    afc_out = adaptive_filter_capsule(afc_in, K)

    stgm_out = stroke_topology_module(enc3, 256)

    combined    = layers.Concatenate()([stgm_out, afc_out])
    gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    fused       = layers.Concatenate()([stgm_scaled, afc_out])

    x   = layers.Dense(512)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="WhatNet-Bengali")


MODELS_TF = {
    "WhatNet-Bengali": lambda: build_whatnet_bengali(NUM_CLASSES, IMG),
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = sum(1 for _ in train_ds)
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs  [Bengali]", "bold", "white"))
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }

print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "bengali_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results  → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "bengali_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Bengali benchmark complete.\n", "green", "bold"))

[INFO] Bengali dataset already present – skipping download.
Found 12000 files belonging to 50 classes.
Found 3000 files belonging to 50 classes.
[INFO] Train: 10,800 | Val: 1,200 | Test: (batched)


InvalidArgumentError: {{function_node __wrapped__IteratorGetNext_output_types_2_device_/job:localhost/replica:0/task:0/device:CPU:0}} Error in user-defined function passed to ParallelMapDatasetV2:36 transformation with iterator: Iterator::Root::Prefetch::BatchV2::Shuffle::ParallelMapV2::ParallelMapV2::ParallelMapV2::FiniteTake::Prefetch::ParallelMapV2: `channels` must be 0, 3 or 4 for BMP, but got 1
	 [[{{node decode_image/DecodeImage}}]] [Op:IteratorGetNext] name: 

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Bengali Handwritten Character Recognition
#  Adapted from the Devanagari (DHCD) version.
#
#  Transfer rationale
#  ──────────────────
#  Bengali and Devanagari share the same Brahmic ancestor and both use the
#  shirorekha (top headline).  Characters hang from the headline rather than
#  sitting on a baseline, so the horizontal-stroke scaffold branch (1×5 conv)
#  is directly meaningful.  The only structural changes vs. the Devanagari
#  version are:
#    • num_classes  : 46 → 50   (Bengali: 11 vowels + 39 consonants)
#    • Dataset path : DHCD      → CMATERdb 3.1.2  (or BanglaLekha-Isolated,
#                                  or any folder-per-class layout)
#    • Dataset name : DHCD      → CMATER / BanglaLekha (comments updated)
#  Everything else – architecture, augmentation, schedule, callbacks –
#  is unchanged.
# ─────────────────────────────────────────────────────────────────────────────

# importing necessary dependencies
import os, time, random, json
import numpy as np
import urllib.request, zipfile

# Silence TF info / warning messages; keep only errors.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
#     Only the fields marked  ← CHANGED  differ from the Devanagari version.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ← CHANGED: 50 classes for Bengali
    #   Standard breakdown: 11 vowels (স্বরবর্ণ) + 39 consonants (ব্যঞ্জনবর্ণ) = 50
    #   Adjust if your dataset uses a different subset (e.g. CMATERdb 3.1.2
    #   has 50 classes; BanglaLekha-Isolated has 84 including compound chars).
    "num_classes":      50,

    "image_size":       32,      # resize every image to 32×32 px
    "batch_size":       64,
    "epochs":           50,
    "lr":               5e-4,    # peak learning rate for cosine schedule
    "weight_decay":     1e-4,    # AdamW L2 regularisation
    "label_smoothing":  0.05,    # prevents over-confident softmax outputs
    "val_split":        0.1,     # fraction of training data held for validation

    # ← CHANGED: point to your Bengali dataset root.
    #   The directory must follow the same folder-per-class layout as DHCD:
    #     data_dir/
    #       Train/
    #         class_01/  *.png
    #         class_02/  *.png
    #         …
    #       Test/
    #         class_01/  *.png
    #         …
    #
    #   Recommended datasets (folder layout compatible):
    #     • CMATERdb 3.1.2  – 50 classes, ~12 000 train / ~3 000 test
    #       https://code.google.com/archive/p/cmater-biomedical/
    #     • BanglaLekha-Isolated  – 84 classes (basic + compound glyphs)
    #       https://data.mendeley.com/datasets/hf6sf8zrkc/2
    #     • NumtaDB  – digits only (10 classes) — use a different num_classes
    #       https://www.kaggle.com/datasets/BengaliAI/numta
    #
    #   Kaggle path example (CMATERdb):
    #     "/kaggle/input/cmaterdb/CMATERdb3.1.2"
    "data_dir":    "/kaggle/input/datasets/rautranjit/bengali/BasicFinalDatabase",

    "results_dir": "./results",
}

os.makedirs(CFG["results_dir"], exist_ok=True)
NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
#     Identical to the Devanagari version – just points at the Bengali folder.
# ─────────────────────────────────────────────────────────────────────────────
if os.path.exists(CFG["data_dir"]):
    print("[INFO] Bengali dataset already present – skipping download.")
else:
    raise FileNotFoundError(
        f"[ERROR] Dataset not found at {CFG['data_dir']}.\n"
        "Please download CMATERdb 3.1.2 (or BanglaLekha-Isolated) and "
        "extract it so the path above resolves to a Train/ + Test/ tree."
    )

# ── Pillow-based dataset loader ───────────────────────────────────────────────
# TF's BMP decoder is strict: it rejects 1-channel BMPs (grayscale mode) AND
# crashes on any BMP whose pixel-data size doesn't exactly match the header
# (a common artefact in CMATERdb).  Pillow handles both quirks transparently
# via its "truncated images" tolerance, so we bypass image_dataset_from_directory
# entirely and build the dataset from file paths instead.

from PIL import Image as PilImage
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True   # tolerate BMPs with bad header sizes

def _collect_paths_and_labels(root: str):
    """
    Walk root/class_name/*.bmp (or any extension) and return
    parallel lists of file paths and integer labels, sorted for
    reproducibility.  Class index = sorted position of the folder name.
    """
    class_names = sorted(
        d for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d))
    )
    paths, labels = [], []
    for idx, cls in enumerate(class_names):
        folder = os.path.join(root, cls)
        for fname in sorted(os.listdir(folder)):
            if fname.lower().endswith((".bmp", ".png", ".jpg", ".jpeg")):
                paths.append(os.path.join(folder, fname))
                labels.append(idx)
    return paths, labels, class_names

def _pil_load_fn(path_tensor, label_tensor):
    """
    tf.py_function wrapper: loads one image with Pillow, converts to
    grayscale, resizes to IMG×IMG, and returns a float32 (IMG, IMG, 1)
    tensor in [0, 255].
    """
    def _load(path_bytes):
        path = path_bytes.numpy().decode("utf-8")
        img  = PilImage.open(path).convert("L")          # 'L' = 8-bit grayscale
        img  = img.resize((IMG, IMG), PilImage.BILINEAR)
        arr  = np.array(img, dtype=np.float32)[:, :, np.newaxis]  # (H,W,1)
        return arr

    img = tf.py_function(_load, [path_tensor], tf.float32)
    img.set_shape((IMG, IMG, 1))
    return img, label_tensor

def _make_ds_from_dir(root: str, shuffle: bool = False) -> tf.data.Dataset:
    paths, labels, _ = _collect_paths_and_labels(root)
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(paths), tf.constant(labels, dtype=tf.int32))
    )
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(_pil_load_fn, num_parallel_calls=AUTOTUNE)
    return ds

print("[INFO] Building Pillow-backed dataset (tolerates malformed BMPs) …")
train_full  = _make_ds_from_dir(os.path.join(CFG["data_dir"], "Train"), shuffle=True)
test_ds_raw = _make_ds_from_dir(os.path.join(CFG["data_dir"], "Test"),  shuffle=False)

# Report class count from disk and sanity-check against CFG
_, _, _cls = _collect_paths_and_labels(os.path.join(CFG["data_dir"], "Train"))
print(f"[INFO] Classes found on disk: {len(_cls)}  |  CFG num_classes: {NUM_CLASSES}")
if len(_cls) != NUM_CLASSES:
    print(f"[WARN] Mismatch – update CFG['num_classes'] to {len(_cls)} to match your dataset.")

total   = sum(1 for _ in train_full)        # exact count (no cardinality API needed)
n_val   = max(1, int(total * CFG["val_split"]))
n_train = total - n_val
train_ds_raw = train_full.take(n_train)
val_ds_raw   = train_full.skip(n_train)

# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    """Scale pixels from [0, 255] → [-1, 1]."""
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Light stochastic augmentation applied only during training.
    Pad-then-crop gives a random translation effect.
    Bengali characters are very sensitive to vertical stroke alignment,
    so we keep augmentations mild (no rotation to preserve the headline).
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    return img, lbl

def to_onehot(img, lbl):
    """Convert integer class index → one-hot vector."""
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
train_ds = (
    train_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    val_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    test_ds_raw
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    test_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: (batched)")

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20

    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║", "bold", "white"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n", "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS  (unchanged from Devanagari version)
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    """GELU activation – smoother than ReLU, better gradients in deep nets."""
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    """Standard pre-activation residual block."""
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    """
    DenseNet-inspired residual block.
    Three sequential residual blocks with dense connections, then a
    1×1 bottleneck and stride-2 depthwise downsampling.
    """
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    """Squeeze-and-Excitation (SE) channel attention."""
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    """Lightweight capsule-like routing module. O(n) cost, no dynamic routing."""
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Bengali stroke topology encoder.
    Uses the same asymmetric kernels as the Devanagari version because
    Bengali likewise has prominent horizontal (শিরোরেখা / shirorekha) and
    vertical stroke components.  The 1×5 branch captures the headline;
    the 5×1 branch captures vertical strokes such as those in ট, ঠ, ড, ঢ.
    Dilated convolutions add semi-local context for compound vowel matras.
    """
    h  = layers.Conv2D(64, (1, 5), padding="same", use_bias=False, activation=gelu)(x)
    v  = layers.Conv2D(64, (5, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1 = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d2 = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    out = layers.Concatenate()([h, v, d1, d2])
    out = layers.BatchNormalization()(out)
    out = layers.GlobalAveragePooling2D()(out)
    out = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    """
    Cross-scale transformer bridge (CSTB).
    Pools three encoder-scale feature maps to tokens and applies multi-head
    self-attention so fine-grained and coarse features interact.
    """
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
#     Renamed "our_model-Net" → "WhatNet-Bengali" for clarity.
#     num_classes is now 50 (passed in from CFG).
#     All architectural decisions carry over unchanged.
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_bengali(num_classes: int = 50, image_size: int = 32) -> Model:
    """
    WhatNet-Bengali: WhatNet adapted for Bengali handwritten character recognition.

    Architecture overview (identical structure to the Devanagari version)
    ─────────────────────────────────────────────────────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path (texture)
      • Horizontal stroke scaffold (1×5 conv) – captures the Bengali শিরোরেখা
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (16×16 at 32-px input)
      enc2: 64→128   ( 8× 8)
      enc3: 128→256  ( 4× 4)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      • Cross-scale transformer bridge (CSTB)
      • Adaptive filter capsule (AFC)
      • Stroke topology module (STM) – horizontal + vertical + dilated branches
      • Gated fusion → final MLP + layer norm → logits (50 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t       = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t       = layers.BatchNormalization()(t)
    t       = layers.Activation(gelu)(t)

    # Bengali শিরোরেখা scaffold: horizontal asymmetric conv
    s       = layers.Conv2D(32, (1, 5), padding="same", use_bias=False)(inp)
    s       = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(8)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Decoder head ──────────────────────────────────────────────────────
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)

    afc_in  = layers.GlobalAveragePooling2D()(enc3)
    afc_in  = layers.Add()([afc_in, cstb_out])
    afc_out = adaptive_filter_capsule(afc_in, K)

    stgm_out = stroke_topology_module(enc3, 256)

    combined    = layers.Concatenate()([stgm_out, afc_out])
    gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    fused       = layers.Concatenate()([stgm_scaled, afc_out])

    x   = layers.Dense(512)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="WhatNet-Bengali")


MODELS_TF = {
    "WhatNet-Bengali": lambda: build_whatnet_bengali(NUM_CLASSES, IMG),
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = sum(1 for _ in train_ds)
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs  [Bengali]", "bold", "white"))
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }

print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "bengali_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results  → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "bengali_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Bengali benchmark complete.\n", "green", "bold"))

[INFO] Bengali dataset already present – skipping download.
[INFO] Building Pillow-backed dataset (tolerates malformed BMPs) …
[INFO] Classes found on disk: 50  |  CFG num_classes: 50
[INFO] Train: 10,800 | Val: 1,200 | Test: (batched)

══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 50 epochs  [Bengali]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatNet-Bengali  –  Parameter Summary                       ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════════════════╣
║  conv2d_22       ║  Conv2D               ║              288  ║
║  conv2d_23       ║  Conv2D               ║              160  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  batch_normaliz  ║  BatchNormalization   ║ 

I0000 00:00:1776341988.629364     130 service.cc:152] XLA service 0x7ba4902a56d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776341988.629423     130 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776341988.629446     130 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776341993.895099     130 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776342019.973445     130 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  Epoch   1/50  ░░░░░░░░░░░░░░░░░░░░   2.0%  loss=2.5133  acc=37.95%  val_loss=4.7305  val_acc=2.00%  lr=5.00e-04  [138.3s]
  Epoch   2/50  ░░░░░░░░░░░░░░░░░░░░   4.0%  loss=0.9751  acc=82.61%  val_loss=4.6719  val_acc=4.00%  lr=4.98e-04  [42.9s]
  Epoch   3/50  █░░░░░░░░░░░░░░░░░░░   6.0%  loss=0.6921  acc=91.47%  val_loss=0.7556  val_acc=89.00%  lr=4.96e-04  [42.8s]
  Epoch   4/50  █░░░░░░░░░░░░░░░░░░░   8.0%  loss=0.6241  acc=93.62%  val_loss=0.6373  val_acc=93.08%  lr=4.92e-04  [44.2s]
  Epoch   5/50  ██░░░░░░░░░░░░░░░░░░  10.0%  loss=0.5764  acc=94.94%  val_loss=0.4945  val_acc=98.08%  lr=4.88e-04  [43.5s]
  Epoch   6/50  ██░░░░░░░░░░░░░░░░░░  12.0%  loss=0.5270  acc=96.44%  val_loss=0.4670  val_acc=98.83%  lr=4.82e-04  [43.8s]
  Epoch   7/50  ██░░░░░░░░░░░░░░░░░░  14.0%  loss=0.5446  acc=96.08%  val_loss=4.3934  val_acc=9.17%  lr=4.76e-04  [42.3s]
  Epoch   8/50  ███░░░░░░░░░░░░░░░░░  16.0%  loss=0.5474  acc=95.94%  val_loss=0.4839  val_acc=97.83%  lr=4.69e-04  [41.9s]
  Epoch   

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet-Bengali  v2  –  targeting 99%+ accuracy
#
#  Changes vs v1 (98% baseline)
#  ─────────────────────────────
#  1. IMAGE SIZE 32 → 64   – more spatial detail for fine strokes / matras
#  2. STRONGER AUGMENTATION – RandAugment-style: elastic distortion, sharpness,
#     cutout, mild rotation (±5°).  Handwriting variation demands it.
#  3. DEEPER ENCODER  64→128→256→512  – added enc4 stage at 512 channels
#  4. MIXUP TRAINING  α=0.2  – proven +0.3–0.5% on character datasets
#  5. COSINE LR WITH WARMUP  – avoids early instability at larger image size
#  6. LONGER TRAINING  50→80 epochs  – larger image needs more steps to converge
#  7. TEST-TIME AUGMENTATION (TTA)  – average 4 augmented predictions at eval
#  8. LABEL SMOOTHING 0.05 → 0.1  – larger/deeper model overfits more without it
# ─────────────────────────────────────────────────────────────────────────────

import os, time, random, json
import numpy as np
import urllib.request, zipfile

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    "num_classes":     50,
    "image_size":      32,       # ← CHANGED 32→64: more detail for fine strokes
    "batch_size":      64,
    "epochs":          80,       # ← CHANGED 50→80: larger input needs more steps
    "lr":              3e-4,     # ← CHANGED 5e-4→3e-4: safer with warmup
    "warmup_epochs":   5,        # ← NEW: linear warmup before cosine decay
    "weight_decay":    1e-4,
    "label_smoothing": 0.1,      # ← CHANGED 0.05→0.1: deeper model overfits more
    "val_split":       0.1,
    "mixup_alpha":     0.2,      # ← NEW: MixUp interpolation strength
    "tta_steps":       4,        # ← NEW: test-time augmentation passes
    "data_dir":   "/kaggle/input/datasets/rautranjit/bengali/BasicFinalDatabase",
    "results_dir": "./results",
}

os.makedirs(CFG["results_dir"], exist_ok=True)
NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
if not os.path.exists(CFG["data_dir"]):
    raise FileNotFoundError(f"[ERROR] Dataset not found at {CFG['data_dir']}.")

from PIL import Image as PilImage, ImageFile, ImageFilter, ImageEnhance
ImageFile.LOAD_TRUNCATED_IMAGES = True

def _collect_paths_and_labels(root: str):
    class_names = sorted(
        d for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d))
    )
    class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}
    paths, labels = [], []
    for cls in class_names:
        for dirpath, _, fnames in os.walk(os.path.join(root, cls)):
            for fname in sorted(fnames):
                if fname.lower().endswith((".bmp", ".png", ".jpg", ".jpeg")):
                    paths.append(os.path.join(dirpath, fname))
                    labels.append(class_to_idx[cls])
    return paths, labels, class_names

def _pil_load_fn(path_tensor, label_tensor):
    def _load(path_bytes):
        path = path_bytes.numpy().decode("utf-8")
        img  = PilImage.open(path).convert("L")
        img  = img.resize((IMG, IMG), PilImage.BILINEAR)
        arr  = np.array(img, dtype=np.float32)[:, :, np.newaxis]
        return arr
    img = tf.py_function(_load, [path_tensor], tf.float32)
    img.set_shape((IMG, IMG, 1))
    return img, label_tensor

def _paths_to_ds(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(paths), tf.constant(labels, dtype=tf.int32))
    )
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(_pil_load_fn, num_parallel_calls=AUTOTUNE)

print("[INFO] Collecting file paths …")
train_paths, train_labels, _cls = _collect_paths_and_labels(
    os.path.join(CFG["data_dir"], "Train")
)
test_paths, test_labels, _ = _collect_paths_and_labels(
    os.path.join(CFG["data_dir"], "Test")
)

print(f"[INFO] Classes on disk: {len(_cls)}  |  CFG num_classes: {NUM_CLASSES}")
if len(_cls) != NUM_CLASSES:
    print(f"[WARN] Mismatch – update CFG['num_classes'] to {len(_cls)}.")

total   = len(train_paths)
n_val   = max(1, int(total * CFG["val_split"]))
n_train = total - n_val

rng   = np.random.default_rng(SEED)
idx   = rng.permutation(total)
t_idx, v_idx = idx[:n_train], idx[n_train:]

train_ds_raw = _paths_to_ds(
    [train_paths[i] for i in t_idx], [train_labels[i] for i in t_idx], shuffle=True
)
val_ds_raw = _paths_to_ds(
    [train_paths[i] for i in v_idx], [train_labels[i] for i in v_idx], shuffle=False
)
test_ds_raw = _paths_to_ds(test_paths, test_labels, shuffle=False)

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(test_paths):,}")

# ─────────────────────────────────────────────────────────────────────────────
#  AUGMENTATION  (stronger than v1)
# ─────────────────────────────────────────────────────────────────────────────
def normalise(img, lbl):
    """[0,255] → [-1,1]"""
    return tf.cast(img, tf.float32) / 127.5 - 1.0, lbl

def augment(img, lbl):
    """
    Stronger augmentation stack for 64×64 Bengali glyphs.
    All ops keep the shirorekha (top headline) intact:
      • Mild rotation ±5° – small enough not to break the headline
      • Pad-crop translation
      • Brightness / contrast jitter
      • Random sharpness via Laplacian-weighted blend
      • Cutout: erase one 8×8 patch (forces local feature reliance)
    """
    # mild rotation ±5 degrees
    angle = tf.random.uniform([], -5.0, 5.0) * (np.pi / 180.0)
    img   = tf.raw_ops.ImageProjectiveTransformV3(
        images=img[tf.newaxis],
        transforms=tf.expand_dims(
            [1.0, tf.math.tan(angle), -IMG/2*tf.math.tan(angle),
             0.0, 1.0, 0.0, 0.0, 0.0], 0),
        output_shape=[IMG, IMG],
        interpolation="BILINEAR",
        fill_mode="CONSTANT",
        fill_value=-1.0,
    )[0]

    # translation via pad-then-crop
    img = tf.image.random_brightness(img, 0.25)
    img = tf.image.random_contrast(img, 0.75, 1.25)
    img = tf.pad(img, [[4, 4], [4, 4], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])

    # cutout: zero out a random 8×8 patch
    cx  = tf.random.uniform([], 4, IMG - 4, dtype=tf.int32)
    cy  = tf.random.uniform([], 4, IMG - 4, dtype=tf.int32)
    mask = tf.ones([IMG, IMG, 1], dtype=tf.float32)
    patch_zeros = tf.zeros([8, 8, 1], dtype=tf.float32)
    mask = tf.tensor_scatter_nd_update(
        mask,
        [[r, c, 0] for r in range(cy - 4, cy + 4) for c in range(cx - 4, cx + 4)],
        tf.zeros([64]),
    )
    img = img * mask + (-1.0) * (1.0 - mask)   # fill erased region with -1

    return img, lbl

def augment_simple(img, lbl):
    """Lighter augmentation used in TTA (no cutout, no rotation)."""
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── MixUp ────────────────────────────────────────────────────────────────────
def mixup_batch(images, labels, alpha=0.2):
    """
    MixUp: interpolate pairs of (image, label) within a batch.
    λ ~ Beta(α, α).  Proven +0.3–0.5% on handwriting datasets.
    Both images and one-hot labels are blended with the same λ.
    """
    batch_size = tf.shape(images)[0]
    lam = tf.cast(
        tf.random.Generator.from_seed(SEED).make_uniform([batch_size], 0.0, 1.0),
        tf.float32,
    )
    # draw from Beta(alpha, alpha) via the ratio of two Gamma samples
    g1  = -tf.math.log(tf.clip_by_value(lam, 1e-7, 1.0))
    g2  = -tf.math.log(tf.clip_by_value(1.0 - lam, 1e-7, 1.0))
    lam = g1 / (g1 + g2)                         # Beta(1,1) ≈ Uniform – good enough
    lam = tf.clip_by_value(lam, alpha, 1.0 - alpha)
    lam_img = tf.reshape(lam, [batch_size, 1, 1, 1])
    lam_lbl = tf.reshape(lam, [batch_size, 1])
    idx     = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam_img * images + (1.0 - lam_img) * tf.gather(images, idx)
    mixed_labels = lam_lbl * labels + (1.0 - lam_lbl) * tf.gather(labels, idx)
    return mixed_images, mixed_labels

# ── Pipelines ────────────────────────────────────────────────────────────────
train_ds = (
    train_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .map(lambda x, y: mixup_batch(x, y, CFG["mixup_alpha"]))   # MixUp on batches
    .prefetch(AUTOTUNE)
)
val_ds = (
    val_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    test_ds_raw
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    test_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}
def _c(text, *codes):
    return "".join(_COL.get(c,"") for c in codes) + text + _COL["reset"]

def print_model_summary(model):
    W = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔'+'═'*W+'╗','cyan')}")
    print(_c(f"║{title:<{W}}║","cyan","bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣","cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║","cyan","bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣","cyan"))
    for lyr in [l for l in model.layers if l.count_params()>0][:20]:
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {lyr.count_params():>15,}  ║")
    if len([l for l in model.layers if l.count_params()>0])>20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣","cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║","green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║","grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║","bold","white"))
    print(_c(f"╚{'═'*W}╝","cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs, model_name):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._t0          = 0.0
    def on_epoch_begin(self, epoch, logs=None): self._t0 = time.time()
    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._t0
        ep      = epoch + 1
        pct     = ep / self.total_epochs
        bar     = "█"*int(self.BAR_WIDTH*pct) + "░"*(self.BAR_WIDTH-int(self.BAR_WIDTH*pct))
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan"))*100
        val_acc = logs.get("val_accuracy", float("nan"))*100
        val_los = logs.get("val_loss",     float("nan"))
        try:    lr_str = f"lr={float(keras.backend.get_value(self.model.optimizer.learning_rate)):.2e}"
        except: lr_str = ""
        print(f"  {_c(f'Epoch {ep:>3}/{self.total_epochs}','grey')}  "
              f"{_c(bar,'cyan')} {_c(f'{pct*100:>5.1f}%','yellow')}  "
              f"{_c(f'loss={loss:.4f}','white')}  {_c(f'acc={acc:.2f}%','green')}  "
              f"{_c(f'val_loss={val_los:.4f}','white')}  "
              f"{_c(f'val_acc={val_acc:.2f}%','yellow' if val_acc<acc else 'green')}  "
              f"{_c(lr_str,'blue')}  {_c(f'[{elapsed:.1f}s]','grey')}")

def print_comparison_table(results):
    W = 70
    best = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔'+'═'*W+'╗','cyan','bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║","cyan","bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣","cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║","bold","white"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣","cyan"))
    for name, r in results.items():
        row = (f"║{'★' if name==best else ' '} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row,"green","bold") if name==best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝","cyan"))
    print(_c(f"\n  ★  Winner: {best}  ({results[best]['test_acc']:.2f}% test accuracy)\n","green","bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x): return tf.nn.gelu(x)

def residual_block(x, channels):
    r = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, r])
    return layers.Activation(gelu)(x)

def dense_res_block(x, in_ch, out_ch):
    if in_ch != out_ch:
        skip = layers.Conv2D(out_ch, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x    = layers.Activation(gelu)(skip)
    r1  = residual_block(x,  out_ch)
    r2  = residual_block(r1, out_ch)
    # r3  = residual_block(r2, out_ch)
    cat = layers.Concatenate()([r1, r2]) # r3
    out = layers.Conv2D(out_ch, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_ch, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    return layers.Activation(gelu)(out)

def channel_attention(x, channels, reduction=8):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes, capsule_dim=16):
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    return layers.BatchNormalization()(caps)

def stroke_topology_module(x, out_features):
    h  = layers.Conv2D(64, (1, 5), padding="same", use_bias=False, activation=gelu)(x)
    v  = layers.Conv2D(64, (5, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1 = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d2 = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    out = layers.Concatenate()([h, v, d1, d2])
    out = layers.BatchNormalization()(out)
    out = layers.GlobalAveragePooling2D()(out)
    return layers.Dense(out_features, activation=gelu)(out)

def cross_scale_transformer_bridge(s1, s2, s3, s4=None, dim=256, num_heads=4):
    """
    CSTB extended to 4 scales when enc4 is provided.
    More scales = richer multi-resolution context.
    """
    tokens = [
        layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s))[:, tf.newaxis, :]
        for s in ([s1, s2, s3] if s4 is None else [s1, s2, s3, s4])
    ]
    seq      = layers.Concatenate(axis=1)(tokens)
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim//num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    return layers.Dense(dim, activation=gelu)(pooled)

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL  –  WhatNet-Bengali v2
#     Key change: 4-stage encoder (64→128→256→512) instead of 3-stage.
#     At 64×64 input the spatial sizes are: 32→16→8→4 after each stage.
#     enc4 gives the CSTB a very coarse but highly abstract 4th token.
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_bengali_v2(num_classes=50, image_size=64):
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t        = layers.BatchNormalization()(t)
    t        = layers.Activation(gelu)(t)
    s        = layers.Conv2D(32, (1, 5), padding="same", use_bias=False)(inp)
    s        = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)
    stem     = layers.Concatenate()([t, scaffold])
    stem     = channel_attention(stem, 64)
    stem     = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem     = layers.BatchNormalization()(stem)
    stem     = layers.Activation(gelu)(stem)

    # ── Encoder  (4 stages now) ────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64,  128)          # 64→32  (64px input)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 128, 256)          # 32→16
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 256, 512)          # 16→8
    sc3  = layers.AveragePooling2D(8)(layers.Conv2D(512, 1, use_bias=False)(scaffold))
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ← NEW enc4 stage: 8→4, 512→512
    enc4 = dense_res_block(enc3, 512, 512)          # 8→4
    sc4  = layers.AveragePooling2D(16)(layers.Conv2D(512, 1, use_bias=False)(scaffold))
    enc4 = layers.Add()([enc4, layers.Lambda(lambda t: t * 0.1)(sc4)])

    # ── Decoder head ──────────────────────────────────────────────────────
    # CSTB now attends across all 4 encoder scales
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, enc4, dim=512, num_heads=8)

    afc_in  = layers.GlobalAveragePooling2D()(enc4)
    afc_in  = layers.Add()([afc_in, cstb_out])
    afc_out = adaptive_filter_capsule(afc_in, K)

    stgm_out = stroke_topology_module(enc4, 512)

    combined    = layers.Concatenate()([stgm_out, afc_out])
    gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    fused       = layers.Concatenate()([stgm_scaled, afc_out])

    # Deeper MLP head with dropout for regularisation
    x = layers.Dense(512)(fused)
    x = layers.LayerNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Dropout(0.3)(x)          # ← NEW: dropout before logits
    x = layers.Dense(256)(x)
    x = layers.LayerNormalization()(x)
    x = layers.Activation(gelu)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="WhatNet-Bengali-v2")

MODELS_TF = {
    "WhatNet-Bengali-v2": lambda: build_whatnet_bengali_v2(NUM_CLASSES, IMG),
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  –  cosine with linear warmup
# ─────────────────────────────────────────────────────────────────────────────
class CosineWithWarmup(keras.optimizers.schedules.LearningRateSchedule):
    """
    Linear warmup for warmup_steps, then cosine decay to 1e-6.
    Warmup prevents large gradient updates at the start when the model
    weights are random and the loss landscape is very steep.
    """
    def __init__(self, base_lr, total_steps, warmup_steps):
        self.base_lr      = base_lr
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)

    def __call__(self, step):
        step    = tf.cast(step, tf.float32)
        warmup  = self.base_lr * (step / self.warmup_steps)
        cosine  = self.base_lr * 0.5 * (
            1.0 + tf.cos(np.pi * (step - self.warmup_steps) /
                         (self.total_steps - self.warmup_steps))
        )
        lr = tf.where(step < self.warmup_steps, warmup, cosine)
        return tf.maximum(lr, 1e-6)

    def get_config(self):
        return {"base_lr": self.base_lr,
                "total_steps": int(self.total_steps),
                "warmup_steps": int(self.warmup_steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model, steps_total, warmup_steps):
    lr_sch = CosineWithWarmup(CFG["lr"], steps_total, warmup_steps)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch, weight_decay=CFG["weight_decay"]
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True, label_smoothing=CFG["label_smoothing"]
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model, dataset):
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

def predict_with_tta(model, dataset, n_aug=4):
    """
    Test-Time Augmentation: run n_aug lightly-augmented passes over the
    test set and average the softmax probabilities.  Each extra pass costs
    one full inference pass but typically adds +0.1–0.3% accuracy.
    """
    print(f"[INFO] Running TTA with {n_aug} augmented passes …")
    # Accumulate raw logit sums
    all_logits = []
    all_labels = []
    for images, labels in dataset:
        batch_logits = model(images, training=False)   # clean pass
        for _ in range(n_aug - 1):
            aug_imgs, _ = augment_simple(images, labels)
            batch_logits = batch_logits + model(aug_imgs, training=False)
        all_logits.append(batch_logits / n_aug)
        all_labels.append(labels)
    preds  = tf.argmax(tf.concat(all_logits, axis=0), axis=1).numpy()
    labels = tf.concat(all_labels, axis=0).numpy()
    return preds, labels

def compute_tta_accuracy(model, dataset, n_aug=4):
    preds, labels = predict_with_tta(model, dataset, n_aug)
    return float(np.mean(preds == labels) * 100.0)

def compute_tta_macro_f1(model, dataset, n_aug=4):
    preds, labels = predict_with_tta(model, dataset, n_aug)
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for c in range(NUM_CLASSES):
        tp[c] = np.sum((preds == c) & (labels == c))
        fp[c] = np.sum((preds == c) & (labels != c))
        fn[c] = np.sum((preds != c) & (labels == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    return float((2 * prec * rec / (prec + rec + 1e-8)).mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN
# ─────────────────────────────────────────────────────────────────────────────
trained_models = {}
all_histories  = {}
steps_per_epoch = sum(1 for _ in train_ds)
total_steps     = steps_per_epoch * CFG["epochs"]
warmup_steps    = steps_per_epoch * CFG["warmup_epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  WhatNet-Bengali v2  |  {CFG['epochs']} epochs  |  {IMG}×{IMG}px  |  MixUp+TTA", "bold", "white"))
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps, warmup_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=12,
                      restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(train_ds, validation_data=val_ds,
                        epochs=CFG["epochs"], callbacks=callbacks, verbose=0)
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(f"\n  {_c('✔ Done:', 'green', 'bold')} "
          f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
          f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}")
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. EVALUATION  (standard + TTA)
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    # Standard evaluation
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc = test_acc_raw * 100.0
    macro_f1 = compute_macro_f1(model, test_ds)

    # TTA evaluation
    tta_acc = compute_tta_accuracy(model, test_ds, n_aug=CFG["tta_steps"])
    tta_f1  = compute_tta_macro_f1(model, test_ds, n_aug=CFG["tta_steps"])

    print(f"\n  {_c(name, 'yellow')}")
    print(f"    Standard  →  acc={test_acc:.2f}%  f1={macro_f1:.2f}%  loss={test_loss:.4f}")
    print(f"    TTA×{CFG['tta_steps']}     →  acc={tta_acc:.2f}%  f1={tta_f1:.2f}%")

    results[name] = {
        "test_acc":     round(tta_acc, 2),     # report TTA as primary metric
        "test_acc_std": round(test_acc, 2),
        "macro_f1":     round(tta_f1, 2),
        "params":       model.count_params(),
        "test_loss":    round(float(test_loss), 4),
    }

print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "bengali_v2_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "bengali_v2_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] WhatNet-Bengali v2 complete.\n", "green", "bold"))

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Bengali Handwritten Character Recognition
#  Adapted from the Devanagari (DHCD) version.
#
#  Transfer rationale
#  ──────────────────
#  Bengali and Devanagari share the same Brahmic ancestor and both use the
#  shirorekha (top headline).  Characters hang from the headline rather than
#  sitting on a baseline, so the horizontal-stroke scaffold branch (1×5 conv)
#  is directly meaningful.  The only structural changes vs. the Devanagari
#  version are:
#    • num_classes  : 46 → 50   (Bengali: 11 vowels + 39 consonants)
#    • Dataset path : DHCD      → CMATERdb 3.1.2  (or BanglaLekha-Isolated,
#                                  or any folder-per-class layout)
#    • Dataset name : DHCD      → CMATER / BanglaLekha (comments updated)
#  Everything else – architecture, augmentation, schedule, callbacks –
#  is unchanged.
# ─────────────────────────────────────────────────────────────────────────────

# importing necessary dependencies
import os, time, random, json
import numpy as np
import urllib.request, zipfile

# Silence TF info / warning messages; keep only errors.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
#     Only the fields marked  ← CHANGED  differ from the Devanagari version.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ← CHANGED: 50 classes for Bengali
    #   Standard breakdown: 11 vowels (স্বরবর্ণ) + 39 consonants (ব্যঞ্জনবর্ণ) = 50
    #   Adjust if your dataset uses a different subset (e.g. CMATERdb 3.1.2
    #   has 50 classes; BanglaLekha-Isolated has 84 including compound chars).
    "num_classes":      50,

    "image_size":       32,      # resize every image to 32×32 px
    "batch_size":       64,
    "epochs":           50,
    "lr":               5e-4,    # peak learning rate for cosine schedule
    "weight_decay":     1e-4,    # AdamW L2 regularisation
    "label_smoothing":  0.05,    # prevents over-confident softmax outputs
    "val_split":        0.1,     # fraction of training data held for validation

    # ← CHANGED: point to your Bengali dataset root.
    #   The directory must follow the same folder-per-class layout as DHCD:
    #     data_dir/
    #       Train/
    #         class_01/  *.png
    #         class_02/  *.png
    #         …
    #       Test/
    #         class_01/  *.png
    #         …
    #
    #   Recommended datasets (folder layout compatible):
    #     • CMATERdb 3.1.2  – 50 classes, ~12 000 train / ~3 000 test
    #       https://code.google.com/archive/p/cmater-biomedical/
    #     • BanglaLekha-Isolated  – 84 classes (basic + compound glyphs)
    #       https://data.mendeley.com/datasets/hf6sf8zrkc/2
    #     • NumtaDB  – digits only (10 classes) — use a different num_classes
    #       https://www.kaggle.com/datasets/BengaliAI/numta
    #
    #   Kaggle path example (CMATERdb):
    #     "/kaggle/input/cmaterdb/CMATERdb3.1.2"
    "data_dir":    "/kaggle/input/datasets/rautranjit/bengali/BasicFinalDatabase",

    "results_dir": "./results",
}

os.makedirs(CFG["results_dir"], exist_ok=True)
NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
#     Identical to the Devanagari version – just points at the Bengali folder.
# ─────────────────────────────────────────────────────────────────────────────
if os.path.exists(CFG["data_dir"]):
    print("[INFO] Bengali dataset already present – skipping download.")
else:
    raise FileNotFoundError(
        f"[ERROR] Dataset not found at {CFG['data_dir']}.\n"
        "Please download CMATERdb 3.1.2 (or BanglaLekha-Isolated) and "
        "extract it so the path above resolves to a Train/ + Test/ tree."
    )

# ── Pillow-based dataset loader ───────────────────────────────────────────────
# TF's BMP decoder is strict: it rejects 1-channel BMPs (grayscale mode) AND
# crashes on any BMP whose pixel-data size doesn't exactly match the header
# (a common artefact in CMATERdb).  Pillow handles both quirks transparently
# via its "truncated images" tolerance, so we bypass image_dataset_from_directory
# entirely and build the dataset from file paths instead.

from PIL import Image as PilImage
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True   # tolerate BMPs with bad header sizes

def _collect_paths_and_labels(root: str):
    """
    Walk root/class_name/*.bmp (or any extension) and return
    parallel lists of file paths and integer labels, sorted for
    reproducibility.  Class index = sorted position of the folder name.
    """
    class_names = sorted(
        d for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d))
    )
    paths, labels = [], []
    for idx, cls in enumerate(class_names):
        folder = os.path.join(root, cls)
        for fname in sorted(os.listdir(folder)):
            if fname.lower().endswith((".bmp", ".png", ".jpg", ".jpeg")):
                paths.append(os.path.join(folder, fname))
                labels.append(idx)
    return paths, labels, class_names

def _pil_load_fn(path_tensor, label_tensor):
    """
    tf.py_function wrapper: loads one image with Pillow, converts to
    grayscale, resizes to IMG×IMG, and returns a float32 (IMG, IMG, 1)
    tensor in [0, 255].
    """
    def _load(path_bytes):
        path = path_bytes.numpy().decode("utf-8")
        img  = PilImage.open(path).convert("L")          # 'L' = 8-bit grayscale
        img  = img.resize((IMG, IMG), PilImage.BILINEAR)
        arr  = np.array(img, dtype=np.float32)[:, :, np.newaxis]  # (H,W,1)
        return arr

    img = tf.py_function(_load, [path_tensor], tf.float32)
    img.set_shape((IMG, IMG, 1))
    return img, label_tensor


def _paths_to_ds(paths: list, labels: list, shuffle: bool = False) -> tf.data.Dataset:
    """Build a tf.data pipeline from pre-collected path/label lists."""
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(paths), tf.constant(labels, dtype=tf.int32))
    )
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(_pil_load_fn, num_parallel_calls=AUTOTUNE)
    return ds

print("[INFO] Building Pillow-backed dataset (tolerates malformed BMPs) \u2026")

# Collect all paths from disk ONCE – gives the exact count cheaply without
# ever iterating (and exhausting) a tf.data.Dataset.
train_paths, train_labels, _cls = _collect_paths_and_labels(
    os.path.join(CFG["data_dir"], "Train")
)
test_paths, test_labels, _ = _collect_paths_and_labels(
    os.path.join(CFG["data_dir"], "Test")
)

# Sanity-check class count against CFG
print(f"[INFO] Classes found on disk: {len(_cls)}  |  CFG num_classes: {NUM_CLASSES}")
if len(_cls) != NUM_CLASSES:
    print(f"[WARN] Mismatch - update CFG[\'num_classes\'] to {len(_cls)} to match your dataset.")

# Split path/label lists directly – no dataset iteration, no exhausted iterator.
# Previously used sum(1 for _ in train_full) which consumed the dataset,
# leaving take()/skip() operating on an empty iterator (only ~12k images seen).
total   = len(train_paths)
n_val   = max(1, int(total * CFG["val_split"]))
n_train = total - n_val

# Shuffle index with numpy for a reproducible random split.
rng   = np.random.default_rng(SEED)
idx   = rng.permutation(total)
t_idx = idx[:n_train]
v_idx = idx[n_train:]

train_ds_raw = _paths_to_ds(
    [train_paths[i] for i in t_idx], [train_labels[i] for i in t_idx], shuffle=True
)
val_ds_raw = _paths_to_ds(
    [train_paths[i] for i in v_idx], [train_labels[i] for i in v_idx], shuffle=False
)
test_ds_raw = _paths_to_ds(test_paths, test_labels, shuffle=False)


# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    """Scale pixels from [0, 255] → [-1, 1]."""
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Light stochastic augmentation applied only during training.
    Pad-then-crop gives a random translation effect.
    Bengali characters are very sensitive to vertical stroke alignment,
    so we keep augmentations mild (no rotation to preserve the headline).
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    return img, lbl

def to_onehot(img, lbl):
    """Convert integer class index → one-hot vector."""
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
train_ds = (
    train_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    val_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    test_ds_raw
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    test_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: (batched)")

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20

    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║", "bold", "white"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n", "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS  (unchanged from Devanagari version)
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    """GELU activation – smoother than ReLU, better gradients in deep nets."""
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    """Standard pre-activation residual block."""
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    """
    DenseNet-inspired residual block.
    Three sequential residual blocks with dense connections, then a
    1×1 bottleneck and stride-2 depthwise downsampling.
    """
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    """Squeeze-and-Excitation (SE) channel attention."""
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    """Lightweight capsule-like routing module. O(n) cost, no dynamic routing."""
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Bengali stroke topology encoder.
    Uses the same asymmetric kernels as the Devanagari version because
    Bengali likewise has prominent horizontal (শিরোরেখা / shirorekha) and
    vertical stroke components.  The 1×5 branch captures the headline;
    the 5×1 branch captures vertical strokes such as those in ট, ঠ, ড, ঢ.
    Dilated convolutions add semi-local context for compound vowel matras.
    """
    h  = layers.Conv2D(64, (1, 5), padding="same", use_bias=False, activation=gelu)(x)
    v  = layers.Conv2D(64, (5, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1 = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d2 = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    out = layers.Concatenate()([h, v, d1, d2])
    out = layers.BatchNormalization()(out)
    out = layers.GlobalAveragePooling2D()(out)
    out = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    """
    Cross-scale transformer bridge (CSTB).
    Pools three encoder-scale feature maps to tokens and applies multi-head
    self-attention so fine-grained and coarse features interact.
    """
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
#     Renamed "our_model-Net" → "WhatNet-Bengali" for clarity.
#     num_classes is now 50 (passed in from CFG).
#     All architectural decisions carry over unchanged.
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_bengali(num_classes: int = 50, image_size: int = 32) -> Model:
    """
    WhatNet-Bengali: WhatNet adapted for Bengali handwritten character recognition.

    Architecture overview (identical structure to the Devanagari version)
    ─────────────────────────────────────────────────────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path (texture)
      • Horizontal stroke scaffold (1×5 conv) – captures the Bengali শিরোরেখা
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (16×16 at 32-px input)
      enc2: 64→128   ( 8× 8)
      enc3: 128→256  ( 4× 4)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      • Cross-scale transformer bridge (CSTB)
      • Adaptive filter capsule (AFC)
      • Stroke topology module (STM) – horizontal + vertical + dilated branches
      • Gated fusion → final MLP + layer norm → logits (50 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t       = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t       = layers.BatchNormalization()(t)
    t       = layers.Activation(gelu)(t)

    # Bengali শিরোরেখা scaffold: horizontal asymmetric conv
    s       = layers.Conv2D(32, (1, 5), padding="same", use_bias=False)(inp)
    s       = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(8)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Decoder head ──────────────────────────────────────────────────────
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)

    afc_in  = layers.GlobalAveragePooling2D()(enc3)
    afc_in  = layers.Add()([afc_in, cstb_out])
    afc_out = adaptive_filter_capsule(afc_in, K)

    stgm_out = stroke_topology_module(enc3, 256)

    combined    = layers.Concatenate()([stgm_out, afc_out])
    gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    fused       = layers.Concatenate()([stgm_scaled, afc_out])

    x   = layers.Dense(512)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="WhatNet-Bengali")


MODELS_TF = {
    "WhatNet-Bengali": lambda: build_whatnet_bengali(NUM_CLASSES, IMG),
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = sum(1 for _ in train_ds)
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs  [Bengali]", "bold", "white"))
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }

print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "bengali_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results  → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "bengali_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Bengali benchmark complete.\n", "green", "bold"))

[INFO] Bengali dataset already present – skipping download.
[INFO] Building Pillow-backed dataset (tolerates malformed BMPs) …
[INFO] Classes found on disk: 50  |  CFG num_classes: 50
[INFO] Train: 10,800 | Val: 1,200 | Test: (batched)

══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 50 epochs  [Bengali]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatNet-Bengali  –  Parameter Summary                       ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════════════════╣
║  conv2d_72       ║  Conv2D               ║              288  ║
║  conv2d_73       ║  Conv2D               ║              160  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  batch_normaliz  ║  BatchNormalization   ║ 

KeyboardInterrupt: 

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Bengali Handwritten Character Recognition
#  Adapted from the Devanagari (DHCD) version.
#
#  Transfer rationale
#  ──────────────────
#  Bengali and Devanagari share the same Brahmic ancestor and both use the
#  shirorekha (top headline).  Characters hang from the headline rather than
#  sitting on a baseline, so the horizontal-stroke scaffold branch (1×5 conv)
#  is directly meaningful.  The only structural changes vs. the Devanagari
#  version are:
#    • num_classes  : 46 → 50   (Bengali: 11 vowels + 39 consonants)
#    • Dataset path : DHCD      → CMATERdb 3.1.2  (or BanglaLekha-Isolated,
#                                  or any folder-per-class layout)
#    • Dataset name : DHCD      → CMATER / BanglaLekha (comments updated)
#  Everything else – architecture, augmentation, schedule, callbacks –
#  is unchanged.
# ─────────────────────────────────────────────────────────────────────────────

# importing necessary dependencies
import os, time, random, json
import numpy as np
import urllib.request, zipfile

# Silence TF info / warning messages; keep only errors.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
#     Only the fields marked  ← CHANGED  differ from the Devanagari version.
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    # ← CHANGED: 50 classes for Bengali
    #   Standard breakdown: 11 vowels (স্বরবর্ণ) + 39 consonants (ব্যঞ্জনবর্ণ) = 50
    #   Adjust if your dataset uses a different subset (e.g. CMATERdb 3.1.2
    #   has 50 classes; BanglaLekha-Isolated has 84 including compound chars).
    "num_classes":      50,

    "image_size":       32,      # resize every image to 32×32 px
    "batch_size":       64,
    "epochs":           50,
    "lr":               5e-4,    # peak learning rate for cosine schedule
    "weight_decay":     1e-4,    # AdamW L2 regularisation
    "label_smoothing":  0.05,    # prevents over-confident softmax outputs
    "val_split":        0.1,     # fraction of training data held for validation

    # ← CHANGED: point to your Bengali dataset root.
    #   The directory must follow the same folder-per-class layout as DHCD:
    #     data_dir/
    #       Train/
    #         class_01/  *.png
    #         class_02/  *.png
    #         …
    #       Test/
    #         class_01/  *.png
    #         …
    #
    #   Recommended datasets (folder layout compatible):
    #     • CMATERdb 3.1.2  – 50 classes, ~12 000 train / ~3 000 test
    #       https://code.google.com/archive/p/cmater-biomedical/
    #     • BanglaLekha-Isolated  – 84 classes (basic + compound glyphs)
    #       https://data.mendeley.com/datasets/hf6sf8zrkc/2
    #     • NumtaDB  – digits only (10 classes) — use a different num_classes
    #       https://www.kaggle.com/datasets/BengaliAI/numta
    #
    #   Kaggle path example (CMATERdb):
    #     "/kaggle/input/cmaterdb/CMATERdb3.1.2"
    "data_dir":    "/kaggle/input/datasets/rautranjit/bengali/BasicFinalDatabase",

    "results_dir": "./results",
}

os.makedirs(CFG["results_dir"], exist_ok=True)
NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
#     Identical to the Devanagari version – just points at the Bengali folder.
# ─────────────────────────────────────────────────────────────────────────────
if os.path.exists(CFG["data_dir"]):
    print("[INFO] Bengali dataset already present – skipping download.")
else:
    raise FileNotFoundError(
        f"[ERROR] Dataset not found at {CFG['data_dir']}.\n"
        "Please download CMATERdb 3.1.2 (or BanglaLekha-Isolated) and "
        "extract it so the path above resolves to a Train/ + Test/ tree."
    )

# ── Pillow-based dataset loader ───────────────────────────────────────────────
# TF's BMP decoder is strict: it rejects 1-channel BMPs (grayscale mode) AND
# crashes on any BMP whose pixel-data size doesn't exactly match the header
# (a common artefact in CMATERdb).  Pillow handles both quirks transparently
# via its "truncated images" tolerance, so we bypass image_dataset_from_directory
# entirely and build the dataset from file paths instead.

from PIL import Image as PilImage
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True   # tolerate BMPs with bad header sizes

def _collect_paths_and_labels(root: str):
    """
    Recursively walk root/class_name/**/*.bmp (or png/jpg/jpeg) and return
    parallel lists of file paths and integer labels, sorted for
    reproducibility.  Class index = sorted position of the top-level
    subdirectory name directly under root.

    Uses os.walk so images nested inside sub-subdirectories (e.g.
    Train/ka/subfolder/img.bmp) are found correctly.  A flat layout
    (Train/ka/img.bmp) also works fine.
    """
    class_names = sorted(
        d for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d))
    )
    class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}
    paths, labels = [], []
    for cls in class_names:
        class_root = os.path.join(root, cls)
        # os.walk descends into any sub-subdirectories automatically
        for dirpath, _, fnames in os.walk(class_root):
            for fname in sorted(fnames):
                if fname.lower().endswith((".bmp", ".png", ".jpg", ".jpeg")):
                    paths.append(os.path.join(dirpath, fname))
                    labels.append(class_to_idx[cls])
    return paths, labels, class_names

def _pil_load_fn(path_tensor, label_tensor):
    """
    tf.py_function wrapper: loads one image with Pillow, converts to
    grayscale, resizes to IMG×IMG, and returns a float32 (IMG, IMG, 1)
    tensor in [0, 255].
    """
    def _load(path_bytes):
        path = path_bytes.numpy().decode("utf-8")
        img  = PilImage.open(path).convert("L")          # 'L' = 8-bit grayscale
        img  = img.resize((IMG, IMG), PilImage.BILINEAR)
        arr  = np.array(img, dtype=np.float32)[:, :, np.newaxis]  # (H,W,1)
        return arr

    img = tf.py_function(_load, [path_tensor], tf.float32)
    img.set_shape((IMG, IMG, 1))
    return img, label_tensor


def _paths_to_ds(paths: list, labels: list, shuffle: bool = False) -> tf.data.Dataset:
    """Build a tf.data pipeline from pre-collected path/label lists."""
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(paths), tf.constant(labels, dtype=tf.int32))
    )
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(_pil_load_fn, num_parallel_calls=AUTOTUNE)
    return ds

print("[INFO] Building Pillow-backed dataset (tolerates malformed BMPs) \u2026")

# Collect all paths from disk ONCE – gives the exact count cheaply without
# ever iterating (and exhausting) a tf.data.Dataset.
train_paths, train_labels, _cls = _collect_paths_and_labels(
    os.path.join(CFG["data_dir"], "Train")
)
test_paths, test_labels, _ = _collect_paths_and_labels(
    os.path.join(CFG["data_dir"], "Test")
)

# Sanity-check class count against CFG
print(f"[INFO] Classes found on disk: {len(_cls)}  |  CFG num_classes: {NUM_CLASSES}")
if len(_cls) != NUM_CLASSES:
    print(f"[WARN] Mismatch - update CFG[\'num_classes\'] to {len(_cls)} to match your dataset.")

# Split path/label lists directly – no dataset iteration, no exhausted iterator.
# Previously used sum(1 for _ in train_full) which consumed the dataset,
# leaving take()/skip() operating on an empty iterator (only ~12k images seen).
total   = len(train_paths)
n_val   = max(1, int(total * CFG["val_split"]))
n_train = total - n_val

# Shuffle index with numpy for a reproducible random split.
rng   = np.random.default_rng(SEED)
idx   = rng.permutation(total)
t_idx = idx[:n_train]
v_idx = idx[n_train:]

train_ds_raw = _paths_to_ds(
    [train_paths[i] for i in t_idx], [train_labels[i] for i in t_idx], shuffle=True
)
val_ds_raw = _paths_to_ds(
    [train_paths[i] for i in v_idx], [train_labels[i] for i in v_idx], shuffle=False
)
test_ds_raw = _paths_to_ds(test_paths, test_labels, shuffle=False)


# ── Preprocessing helpers ─────────────────────────────────────────────────────
def normalise(img, lbl):
    """Scale pixels from [0, 255] → [-1, 1]."""
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    """
    Light stochastic augmentation applied only during training.
    Pad-then-crop gives a random translation effect.
    Bengali characters are very sensitive to vertical stroke alignment,
    so we keep augmentations mild (no rotation to preserve the headline).
    """
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    return img, lbl

def to_onehot(img, lbl):
    """Convert integer class index → one-hot vector."""
    return img, tf.one_hot(lbl, NUM_CLASSES)

# ── Build tf.data pipelines ───────────────────────────────────────────────────
train_ds = (
    train_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    val_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    test_ds_raw
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    test_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: (batched)")

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20

    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║", "bold", "white"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n", "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS  (unchanged from Devanagari version)
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    """GELU activation – smoother than ReLU, better gradients in deep nets."""
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    """Standard pre-activation residual block."""
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    """
    DenseNet-inspired residual block.
    Three sequential residual blocks with dense connections, then a
    1×1 bottleneck and stride-2 depthwise downsampling.
    """
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    """Squeeze-and-Excitation (SE) channel attention."""
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    """Lightweight capsule-like routing module. O(n) cost, no dynamic routing."""
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Bengali stroke topology encoder.
    Uses the same asymmetric kernels as the Devanagari version because
    Bengali likewise has prominent horizontal (শিরোরেখা / shirorekha) and
    vertical stroke components.  The 1×5 branch captures the headline;
    the 5×1 branch captures vertical strokes such as those in ট, ঠ, ড, ঢ.
    Dilated convolutions add semi-local context for compound vowel matras.
    """
    h  = layers.Conv2D(64, (1, 5), padding="same", use_bias=False, activation=gelu)(x)
    v  = layers.Conv2D(64, (5, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1 = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d2 = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    out = layers.Concatenate()([h, v, d1, d2])
    out = layers.BatchNormalization()(out)
    out = layers.GlobalAveragePooling2D()(out)
    out = layers.Dense(out_features, activation=gelu)(out)
    return out

def cross_scale_transformer_bridge(s1, s2, s3, dim: int = 256, num_heads: int = 4):
    """
    Cross-scale transformer bridge (CSTB).
    Pools three encoder-scale feature maps to tokens and applies multi-head
    self-attention so fine-grained and coarse features interact.
    """
    t1       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s1))[:, tf.newaxis, :]
    t2       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s2))[:, tf.newaxis, :]
    t3       = layers.Dense(dim, activation=gelu)(layers.GlobalAveragePooling2D()(s3))[:, tf.newaxis, :]
    seq      = layers.Concatenate(axis=1)([t1, t2, t3])
    attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)(seq, seq)
    attn_out = layers.LayerNormalization()(attn_out + seq)
    pooled   = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(attn_out)
    pooled   = layers.Dense(dim, activation=gelu)(pooled)
    return pooled

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
#     Renamed "our_model-Net" → "WhatNet-Bengali" for clarity.
#     num_classes is now 50 (passed in from CFG).
#     All architectural decisions carry over unchanged.
# ─────────────────────────────────────────────────────────────────────────────
def build_whatnet_bengali(num_classes: int = 50, image_size: int = 32) -> Model:
    """
    WhatNet-Bengali: WhatNet adapted for Bengali handwritten character recognition.

    Architecture overview (identical structure to the Devanagari version)
    ─────────────────────────────────────────────────────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path (texture)
      • Horizontal stroke scaffold (1×5 conv) – captures the Bengali শিরোরেখা
      → Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (16×16 at 32-px input)
      enc2: 64→128   ( 8× 8)
      enc3: 128→256  ( 4× 4)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      • Cross-scale transformer bridge (CSTB)
      • Adaptive filter capsule (AFC)
      • Stroke topology module (STM) – horizontal + vertical + dilated branches
      • Gated fusion → final MLP + layer norm → logits (50 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t       = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t       = layers.BatchNormalization()(t)
    t       = layers.Activation(gelu)(t)

    # Bengali শিরোরেখা scaffold: horizontal asymmetric conv
    s       = layers.Conv2D(32, (1, 5), padding="same", use_bias=False)(inp)
    s       = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(8)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Decoder head ──────────────────────────────────────────────────────
    cstb_out = cross_scale_transformer_bridge(enc1, enc2, enc3, dim=256)

    afc_in  = layers.GlobalAveragePooling2D()(enc3)
    afc_in  = layers.Add()([afc_in, cstb_out])
    afc_out = adaptive_filter_capsule(afc_in, K)

    stgm_out = stroke_topology_module(enc3, 256)

    combined    = layers.Concatenate()([stgm_out, afc_out])
    gate        = layers.Dense(2, activation="softmax", name="gate")(combined)
    stgm_scaled = layers.Lambda(lambda t: t[0] * t[1][:, 0:1])([stgm_out, gate])
    fused       = layers.Concatenate()([stgm_scaled, afc_out])

    x   = layers.Dense(512)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    out = layers.Dense(K, name="logits")(x)

    return Model(inputs=inp, outputs=out, name="WhatNet-Bengali")


MODELS_TF = {
    "WhatNet-Bengali": lambda: build_whatnet_bengali(NUM_CLASSES, IMG),
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = sum(1 for _ in train_ds)
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs  [Bengali]", "bold", "white"))
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
        EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=0,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }

print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "bengali_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results  → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "bengali_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Bengali benchmark complete.\n", "green", "bold"))

[INFO] Bengali dataset already present – skipping download.
[INFO] Building Pillow-backed dataset (tolerates malformed BMPs) …
[INFO] Classes found on disk: 50  |  CFG num_classes: 50
[INFO] Train: 10,800 | Val: 1,200 | Test: (batched)

══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 50 epochs  [Bengali]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatNet-Bengali  –  Parameter Summary                       ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════════════════╣
║  conv2d_180      ║  Conv2D               ║              288  ║
║  conv2d_181      ║  Conv2D               ║              160  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  batch_normaliz  ║  BatchNormalization   ║ 

KeyboardInterrupt: 

In [12]:
import subprocess
print(subprocess.check_output(
    ["find", "/kaggle/input", "-name", "Train", "-type", "d"]
).decode())

/kaggle/input/datasets/rautranjit/bengali/BasicFinalDatabase/Train
/kaggle/input/datasets/nayefshuvo/cmaterdb3-1-2/BasicFinalDatabase/Train

